In [ ]:
# CONFIG — all user-editable settings
CONFIG = {
    "project_dir": "/path/to/analysis_home/Projects/scRhythmBayes_simulation",
    "scritmo_repo": "/path/to/analysis_home/Projects/scRhythmBayes_simulation/scRITMO",
    "clock_parameters": "ccg_zhang_context.csv",  # relative to scritmo_repo
    "output_dir": "output/scRITMO_extended_simulation",  # relative to project_dir
    "run_name": "sigma_gradient_1p5_2p5_3p5_4p5_5p5_6p5_empirical_fixed_v1",  # choose a new name for each retained run
    # 20260909 produced CT5 cSD=5.79575 h (error=0.29575 h), failing the frozen 0.25 h gate.
    # 20260910 is the first consecutive replacement seed tested; all six phase-dispersion targets pass.
    "seed": 20260910,
    "period": 24.0,
    "zt_points": [0, 4, 8, 12, 16, 20],
    "n_replicates": 3,  # our extension: independent animals per ZT
    "n_cells_per_sample": 500,
    "celltypes": {
        "CT1": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 1.5},
        "CT2": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 2.5},
        "CT3": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 3.5},
        "CT4": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 4.5},
        "CT5": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 5.5},
        "CT6": {"probability": 1/6, "phase_offset_hour": 0.0, "sigma_bio_hour": 6.5},
    },
    # Optional shared animal-level phase shift, separate from within-sample desynchrony.
    "sample_phase_sd_hour": 0.0,
    "gse145197": {
        "format": "genes_by_cells_tsv_gz",
        "paths": [
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308343_UMI_tab_ZT00A.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308344_UMI_tab_ZT00B.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308345_UMI_tab_ZT00C.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308346_UMI_tab_ZT06A.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308347_UMI_tab_ZT06B.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308348_UMI_tab_ZT12A.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308349_UMI_tab_ZT12B.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308350_UMI_tab_ZT12C.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308351_UMI_tab_ZT18A.txt.gz",
            "/path/to/analysis_home/scRhythmBayes/data/GSE145197_RAW/GSM4308352_UMI_tab_ZT18B.txt.gz"
        ],
        "confirmed_full_gene_raw_counts": True,
        "provenance": "GSE145197 raw full-gene UMI count tables supplied by the user from GEO (GSM4308343-GSM4308352); genes x cells; full-gene column totals pooled across 10 samples without normalization or gene selection. Zero-total cells are excluded and recorded.",
        "read_chunk_genes": 1000
    },
    "genome_parameters": {
        "path": "simulation/empirical_gene_parameters_GSE145197.csv",
        "manifest_path": "simulation/empirical_gene_parameters_GSE145197.manifest.json",
        "provenance": "GSE145197_sample_NB2_cell_MoM_v1: 10 biological GSM sample-pseudobulk harmonic NB fits with full-library log offsets and single-cell within-sample NB2 dispersion. Hash-verified empirical table contains 2985 selected extra genes; this 1000-gene benchmark retains the first 985 rows in file order, not a new stratified selection. Frozen clock genes remain unchanged.",
        "confirmed_full_library_log_offset": True,
        "n_genes_total": 1000,  # includes 15 frozen clock genes
        "allow_development_fallback": False,
        "development_rhythmic_fraction": 0.2,
        "development_mesor_sd": 0.3,
    },
    # Optional CSV: gene_id,celltype,delta_mesor,amplitude_scale,
    # amplitude_add,phase_shift_hour. Missing gene/type rows mean no perturbation.
    "perturbations_path": None,
    "perturbations_provenance": None,
    "original": {"n_cells": 1000, "seq_depth": 1000.0},
    "cell_chunk": 512,  # CPU generation; affects RNG consumption, retained in manifest
    "qc": {"bins": 40, "trajectory_genes": ["Bmal1", "Per2", "Dbp"],
           "trajectory_cells": 1000, "dpi": 140,
           "phase_sigma_tolerance_hour": 0.25},
}

In [ ]:
import torch
import numpy as np
import pandas as pd
import anndata
from scipy import sparse
from scipy.optimize import brentq
from scipy.special import i0e, i1e
from scipy.stats import circstd, ks_2samp
from matplotlib import pyplot as plt

import scritmo as sr
from scritmo.ml.simulations import simulations, simulate_populations
from scritmo.ml.simulations.simulations import simulate_data_no_context

import hashlib
import re
import inspect
import itertools
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from IPython.display import display

print("Python:", sys.executable)
print("scRITMO:", sr.__file__)
print("Torch:", torch.__version__)

In [ ]:
SOURCE_HASHES = {
    "simulations.py": "e830d6582f3b861c7435b0ee35552cfa813bba0a4ebf1f3df636da880f642000",
    "simulate_populations.py": "b59e3860cc1309e04eecc0243e84e948f06e4737d0da6b1f7d232e0d23ac5966",
}
CLOCK_HASH = "395f3e37cf4034d319266018633f80307e66b83f79cc38bcea3562380f164182"
CLOCK_GENES = ["Bhlhe40", "Bhlhe41", "Bmal1", "Ciart", "Clock", "Cry1", "Dbp",
               "Npas2", "Nr1d1", "Nr1d2", "Per1", "Per2", "Per3", "Rorc", "Tef"]
PARAM_COLS = ["a_0", "a_1", "b_1", "disp"]


def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def project_path(value):
    p = Path(value).expanduser()
    return p if p.is_absolute() else Path(CONFIG["project_dir"]).expanduser() / p


repo = Path(CONFIG["scritmo_repo"]).expanduser().resolve()
source_info = {}
for module in [simulations, simulate_populations]:
    path = Path(inspect.getfile(module)).resolve()
    assert path.is_relative_to(repo), f"Wrong scRITMO checkout: {path}"
    actual_hash = file_sha256(path)
    assert actual_hash == SOURCE_HASHES[path.name], f"Source changed: {path}"
    source_info[path.name] = {"path": str(path), "sha256": actual_hash}

scritmo_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
scritmo_status = subprocess.check_output(
    ["git", "-C", str(repo), "status", "--porcelain"], text=True
).strip()
clock_path = repo / CONFIG["clock_parameters"]
assert file_sha256(clock_path) == CLOCK_HASH, "Reference parameter file changed."
params_g_clock = pd.read_csv(clock_path, index_col=0)
assert params_g_clock.index.tolist() == CLOCK_GENES
assert np.isfinite(params_g_clock[PARAM_COLS].to_numpy()).all()
assert (params_g_clock["disp"] > 0).all()

assert CONFIG["zt_points"] == [0, 4, 8, 12, 16, 20]
assert CONFIG["period"] == 24.0
for key in ["n_replicates", "n_cells_per_sample", "cell_chunk"]:
    assert isinstance(CONFIG[key], int) and CONFIG[key] > 0
assert CONFIG["n_replicates"] >= 2, "Use explicit biological replicates."
assert isinstance(CONFIG["seed"], int) and 0 <= CONFIG["seed"] < 2**32
assert np.isfinite(CONFIG["sample_phase_sd_hour"]) and CONFIG["sample_phase_sd_hour"] >= 0
celltype_names = list(CONFIG["celltypes"])
probabilities = np.array([CONFIG["celltypes"][ct]["probability"] for ct in celltype_names])
assert len(celltype_names) >= 2 and np.isfinite(probabilities).all()
assert (probabilities > 0).all() and np.isclose(probabilities.sum(), 1.0)
for setting in CONFIG["celltypes"].values():
    assert np.isfinite(setting["phase_offset_hour"])
    assert np.isfinite(setting["sigma_bio_hour"]) and setting["sigma_bio_hour"] >= 0

# Frozen controlled experiment: only sigma_bio varies across cell types.
expected_sigma_bio_hour = {f"CT{i}": sigma for i, sigma in
                           enumerate([1.5, 2.5, 3.5, 4.5, 5.5, 6.5], start=1)}
assert celltype_names == list(expected_sigma_bio_hour)
assert np.allclose(probabilities, 1 / len(celltype_names))
assert CONFIG["perturbations_path"] is None, "Dispersion-only design requires identical gene parameters."
assert {ct: CONFIG["celltypes"][ct]["sigma_bio_hour"] for ct in celltype_names} == expected_sigma_bio_hour
assert all(CONFIG["celltypes"][ct]["phase_offset_hour"] == 0 for ct in celltype_names)
assert CONFIG["sample_phase_sd_hour"] == 0
assert CONFIG["n_replicates"] == 3 and CONFIG["n_cells_per_sample"] == 500
assert CONFIG["genome_parameters"]["n_genes_total"] == 1000

seed_names = ["original", "allocation", "library", "sample_phase", "cell_phase", "parameters", "counts"]
seed_children = np.random.SeedSequence(CONFIG["seed"]).spawn(len(seed_names))
seeds = {name: int(child.generate_state(1)[0]) for name, child in zip(seed_names, seed_children)}
w = 2 * np.pi / CONFIG["period"]
rh = 1 / w
run_dir = project_path(CONFIG["output_dir"]) / CONFIG["run_name"]
run_dir.mkdir(parents=True, exist_ok=True)
display(params_g_clock)

In [ ]:
torch.manual_seed(seeds["original"])
phi_sim_original = torch.linspace(0, 2 * np.pi, CONFIG["original"]["n_cells"]).numpy()
seq_depths_original = np.full(len(phi_sim_original), CONFIG["original"]["seq_depth"])
data_original = simulate_data_no_context(
    phases=phi_sim_original,
    seq_depths=seq_depths_original,
    fourier_coefficients=params_g_clock,
    context_label="Clock-Reference",
    noise_model="nb",
    dispersion=params_g_clock["disp"].to_numpy(),
)
assert data_original.shape == (len(phi_sim_original), 15)
assert np.isfinite(data_original).all() and (data_original >= 0).all()
assert np.equal(data_original, np.floor(data_original)).all()
adata_original = anndata.AnnData(
    X=sparse.csr_matrix(data_original.astype(np.int64)),
    obs=pd.DataFrame({"true_phase_rad": phi_sim_original,
                      "library_size": seq_depths_original},
                     index=[f"original_{i:06d}" for i in range(len(phi_sim_original))]),
    var=params_g_clock.copy(),
)
adata_original.uns["provenance"] = json.dumps({
    "section": "scRITMO-original reproduction", "scRITMO_commit": scritmo_commit,
    "sources": source_info, "clock_sha256": CLOCK_HASH,
    "seed": seeds["original"], "config": CONFIG["original"],
    "dispersion_source": "unchanged reference CSV disp column",
})
original_dir = run_dir / "original"
original_dir.mkdir(exist_ok=True)
adata_original.write_h5ad(original_dir / "scRITMO_original.h5ad")
params_g_clock.to_csv(original_dir / "clock_parameters_unchanged.csv")
(original_dir / "manifest.json").write_text(adata_original.uns["provenance"])
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(phi_sim_original * rh, data_original[:, CLOCK_GENES.index("Bmal1")], s=8, alpha=0.4)
ax.set(xlabel="True phase (hour)", ylabel="Bmal1 count", title="scRITMO-original: 15 genes")
fig.tight_layout()
fig.savefig(original_dir / "original_trajectory.png", dpi=CONFIG["qc"]["dpi"])
plt.show()

In [ ]:
template_dir = run_dir / "templates"
template_dir.mkdir(exist_ok=True)
pd.DataFrame(columns=["gene_id", *PARAM_COLS, "parameter_source"]).to_csv(
    template_dir / "empirical_gene_parameters_TEMPLATE.csv", index=False
)
pd.DataFrame(columns=["gene_id", "celltype", "delta_mesor", "amplitude_scale",
                      "amplitude_add", "phase_shift_hour"]).to_csv(
    template_dir / "celltype_perturbations_TEMPLATE.csv", index=False
)


def read_library_size_pool():
    setting = CONFIG["gse145197"]
    if setting["format"] != "genes_by_cells_tsv_gz":
        raise ValueError("Unsupported GSE145197 input format.")
    if not setting["paths"]:
        raise ValueError("Fill CONFIG['gse145197']['paths'] with raw UMI table paths.")
    if not setting["confirmed_full_gene_raw_counts"] or not setting["provenance"]:
        raise ValueError("Confirm full-gene raw counts and provide GSE145197 provenance in CONFIG.")
    chunk_size = setting["read_chunk_genes"]
    assert isinstance(chunk_size, int) and chunk_size > 0
    pools, file_records = [], []
    gene_panel_hash = None
    for value in setting["paths"]:
        path = project_path(value)
        match = re.fullmatch(r"(GSM\d+)_UMI_tab_(ZT(\d{2})([A-Za-z]))\.txt\.gz", path.name)
        if match is None:
            raise ValueError(f"Unexpected GSE145197 filename: {path.name}")
        accession, source_sample, source_zt, source_replicate = match.groups()
        reader = pd.read_csv(path, sep="\t", index_col=0, compression="gzip", chunksize=chunk_size)
        cell_ids, library_sizes, gene_hash = None, None, hashlib.sha256()
        n_genes = 0
        for chunk in reader:
            current_ids = chunk.columns.astype(str).to_numpy()
            if cell_ids is None:
                cell_ids = current_ids
                library_sizes = np.zeros(len(cell_ids), dtype=np.float64)
            elif not np.array_equal(current_ids, cell_ids):
                raise ValueError(f"Cell columns changed while reading {path.name}.")
            values = chunk.to_numpy()
            if not (np.isfinite(values).all() and (values >= 0).all() and np.equal(values, np.floor(values)).all()):
                raise ValueError(f"{path.name} must contain nonnegative integer raw counts.")
            library_sizes += values.sum(axis=0, dtype=np.float64)
            gene_hash.update("\n".join(chunk.index.astype(str)).encode())
            gene_hash.update(b"\n")
            n_genes += len(chunk)
        if cell_ids is None or n_genes == 0:
            raise ValueError(f"No count data found in {path.name}.")
        current_gene_hash = gene_hash.hexdigest()
        if gene_panel_hash is None:
            gene_panel_hash = current_gene_hash
        elif current_gene_hash != gene_panel_hash:
            raise ValueError("GSE145197 input files do not share the same ordered gene panel.")
        keep = np.isfinite(library_sizes) & (library_sizes > 0)
        pools.append(pd.DataFrame({"source_accession": accession, "source_sample": source_sample, "source_zt": int(source_zt), "source_replicate": source_replicate, "source_cell_id": cell_ids[keep], "library_size": library_sizes[keep]}))
        file_records.append({"path": str(path), "sha256": file_sha256(path), "accession": accession, "source_sample": source_sample, "source_zt": int(source_zt), "source_replicate": source_replicate, "n_genes": n_genes, "gene_panel_sha256": current_gene_hash, "n_cells": len(cell_ids), "n_positive_cells": int(keep.sum()), "n_zero_cells_excluded": int((~keep).sum())})
    pool = pd.concat(pools, ignore_index=True)
    pool.insert(0, "source_pool_row", np.arange(len(pool)))
    if pool.duplicated(["source_accession", "source_cell_id"]).any():
        raise ValueError("Duplicate source-cell identity across GSE145197 files.")
    if pool.empty:
        raise ValueError("No positive full-library totals found.")
    provenance = {"format": setting["format"], "notes": setting["provenance"], "n_files": len(file_records), "n_cells": len(pool), "gene_panel_sha256": gene_panel_hash, "files": file_records}
    return pool, provenance


library_pool, library_provenance = read_library_size_pool()
display(library_pool["library_size"].describe())

## 4. Sample design, random cell types and Von Mises phases

Every ZT has `n_replicates` independent sample/animal IDs. A replicate number denotes position within ZT, not a longitudinally measured animal. Cells are assigned independently within each sample using fixed probabilities (multinomial composition, no forced balancing). Empty sample × cell-type groups are permitted and reported in QC.

For cell type k in sample s, `θ ~ VM(ZT×w + offset_k×w + sample_shift_s×w, κ_k)`. `sigma_bio_hour` is the **circular standard deviation** of residual phase within a sample, not a linear normal SD. We numerically invert `sqrt(-2 log(I1(κ)/I0(κ))) = sigma_bio_hour×w`; zero gives perfectly synchronized cells. This inversion is **our extension** using scaled Bessel functions. The installed scRITMO inverse is a fitted power law with ambiguous unit documentation, so it is not silently used as an exact conversion. The Von Mises sampling itself follows `simulate_populations.py`.

The optional normal animal shift is shared by that sample's cells. At its default zero, replicates have independent draws but no additional between-animal random effect. Cell-type phase offsets and intrinsic gene phases are saved separately; expected peaks against external ZT are also reported. Library-size draws use a separate RNG stream and never consult phase or cell type.

The six-cell-type design uses equal 1/6 target abundance, zero cell-type phase offset, zero sample shift and circular SDs of 1.5, 2.5, 3.5, 4.5, 5.5 and 6.5 h for CT1–CT6. Total size remains 18 samples × 500 = 9000 cells; composition is random, not forced to exactly 1500 cells per type. All cell types share identical gene parameters. No other simulation factor is changed.


In [ ]:
def sigma_hour_to_kappa(sigma_hour):
    if sigma_hour == 0:
        return np.inf
    target = np.exp(-0.5 * (sigma_hour * w)**2)
    if target == 0:
        return 0.0
    if target >= 1:
        raise ValueError("sigma_bio_hour is too small for numerical inversion; use 0 or a larger value.")
    upper = 1.0
    while i1e(upper) / i0e(upper) < target:
        upper *= 2
    return brentq(lambda k: i1e(k) / i0e(k) - target, 0, upper,
                  xtol=np.finfo(float).tiny, rtol=1e-12)


def make_cell_design(library_pool):
    rng_allocation = np.random.default_rng(seeds["allocation"])
    rng_library = np.random.default_rng(seeds["library"])
    rng_sample = np.random.default_rng(seeds["sample_phase"])
    rng_phase = np.random.default_rng(seeds["cell_phase"])
    rows, cells = [], []
    for zt in CONFIG["zt_points"]:
        for replicate in range(1, CONFIG["n_replicates"] + 1):
            sample_name = f"ZT{zt:02d}_rep{replicate:02d}"
            sample_shift = rng_sample.normal(0, CONFIG["sample_phase_sd_hour"])
            n_cells = CONFIG["n_cells_per_sample"]
            rows.append({"sample_name": sample_name, "animal_id": sample_name,
                         "ZT": zt, "replicate": replicate, "n_cells": n_cells,
                         "sample_phase_shift_hour": sample_shift,
                         "design_source": "our extension"})
            celltypes = rng_allocation.choice(celltype_names, size=n_cells, p=probabilities)
            source_idx = rng_library.integers(0, len(library_pool), size=n_cells)
            df = pd.DataFrame({"cell_id": [f"{sample_name}_cell{i:05d}" for i in range(n_cells)],
                               "sample_name": sample_name, "animal_id": sample_name,
                               "ZT": zt, "replicate": replicate, "celltype": celltypes,
                               "sample_phase_shift_hour": sample_shift})
            df["library_size"] = library_pool.iloc[source_idx]["library_size"].to_numpy()
            df["library_source_row"] = library_pool.iloc[source_idx]["source_pool_row"].to_numpy()
            df["library_source_cell_id"] = library_pool.iloc[source_idx]["source_cell_id"].to_numpy()
            for ct in celltype_names:
                mask = df["celltype"] == ct
                setting = CONFIG["celltypes"][ct]
                kappa = sigma_hour_to_kappa(setting["sigma_bio_hour"])
                mean_phase = (zt + setting["phase_offset_hour"] + sample_shift) * w
                n = int(mask.sum())
                eps_phi = np.zeros(n) if np.isinf(kappa) else rng_phase.vonmises(0, kappa, size=n)
                df.loc[mask, "true_phase_rad"] = (mean_phase + eps_phi) % (2 * np.pi)
                df.loc[mask, "phase_center_rad"] = mean_phase % (2 * np.pi)
                df.loc[mask, "phase_offset_hour"] = setting["phase_offset_hour"]
                df.loc[mask, "sigma_bio_hour"] = setting["sigma_bio_hour"]
                df.loc[mask, "kappa"] = kappa
            cells.append(df)
    cell_truth = pd.concat(cells, ignore_index=True)
    cell_truth["true_phase_hour"] = cell_truth["true_phase_rad"] * rh
    cell_truth["library_source"] = "empirical: GSE145197 pooled full-library counts"
    cell_truth["phase_source"] = "our extension: Von Mises around sample ZT"
    return pd.DataFrame(rows), cell_truth


sample_design, cell_truth = make_cell_design(library_pool)
assert cell_truth["cell_id"].is_unique
assert len(cell_truth) == len(sample_design) * CONFIG["n_cells_per_sample"]
display(sample_design)

## 4A. Pre-count ground-truth phase QC — required gate

This check runs immediately after phase sampling and before genome parameters or counts are generated. It verifies that centered true phases become progressively broader from CT1 to CT6 and that the empirical circular SD of each generated cell type is within the pre-specified tolerance of 1.5, 2.5, 3.5, 4.5, 5.5 and 6.5 h. Failure stops the notebook before count generation.


In [ ]:
qc_dir = run_dir / "qc"
qc_dir.mkdir(exist_ok=True)
phase_design_rows = []
fig, ax = plt.subplots(figsize=(7, 4))
for ct in celltype_names:
    df = cell_truth.loc[cell_truth["celltype"] == ct]
    residual_rad = (df["true_phase_rad"].to_numpy() - df["phase_center_rad"].to_numpy() + np.pi) % (2 * np.pi) - np.pi
    observed_csd_hour = float(circstd(residual_rad, high=np.pi, low=-np.pi) * rh)
    target_csd_hour = float(CONFIG["celltypes"][ct]["sigma_bio_hour"])
    phase_design_rows.append({"celltype": ct, "n_cells": len(df), "target_sigma_bio_hour": target_csd_hour, "observed_true_cSD_hour": observed_csd_hour, "absolute_error_hour": abs(observed_csd_hour - target_csd_hour), "mean_centered_offset_hour": float(np.angle(np.mean(np.exp(1j * residual_rad))) * rh)})
    ax.hist(residual_rad * rh, bins=np.linspace(-12, 12, CONFIG["qc"]["bins"] + 1), histtype="step", density=True, linewidth=1.5, label=f"{ct}: {target_csd_hour:g} h")
phase_design_qc = pd.DataFrame(phase_design_rows)
phase_design_qc.to_csv(qc_dir / "00_pre_count_phase_dispersion.csv", index=False)
ax.set(xlabel="True phase minus ZT center (h)", ylabel="Density", title="Ground-truth cell-type phase dispersion")
ax.legend(title="Target sigma_bio")
fig.tight_layout()
fig.savefig(qc_dir / "00_pre_count_phase_dispersion.png", dpi=CONFIG["qc"]["dpi"], bbox_inches="tight")
plt.show()
display(phase_design_qc)
assert (phase_design_qc["n_cells"] > 0).all(), "A cell type received zero cells."
assert phase_design_qc["observed_true_cSD_hour"].is_monotonic_increasing, f"Generated true phase dispersions are not ordered {' < '.join(celltype_names)}."
# If CT5/CT6 fail the unchanged 0.25 h tolerance, reassess finite-sample
# circular-SD variability before deciding on a revised design; do not bypass this gate.
failed_phase_qc = phase_design_qc.loc[
    phase_design_qc["absolute_error_hour"] > CONFIG["qc"]["phase_sigma_tolerance_hour"],
    ["celltype", "n_cells", "target_sigma_bio_hour",
     "observed_true_cSD_hour", "absolute_error_hour"],
]
assert failed_phase_qc.empty, ("Ground-truth phase cSD missed its target; stop before count generation.\n"
                               + failed_phase_qc.to_string(index=False))
print("PRE-COUNT PHASE QC PASSED; count generation may proceed.")

## 4B. Single-cell true phase minus sample ZT — sample × celltype QC

Uses simulation ground truth only: `delta_true_vs_ZT_hour = ((true_phase_hour - ZT + 12) % 24) - 12` in [-12, 12). The existing generator defines the phase center as ZT + celltype phase offset + sample phase shift (modulo 24 h). **Neither offset nor sample shift is subtracted here.** The original centered-dispersion gate above is unchanged.

One summary row represents all cells in one sample × celltype (a phase-distribution summary, not expression-count pseudobulk). Circular mean and SD use the mean resultant vector; median, quartiles and mean absolute delta use the wrapped values on [-12, 12), so their interpretation depends on this cut when distributions cross the boundary. Empty design groups have n_cells=0 and undefined statistics. Zero resultant length gives undefined circular mean and infinite circular SD.

Both plots use common histogram bins and unit-area step densities, avoiding smoothing or circular-boundary artifacts. Version 1 has one facet per celltype and a labeled line per sample, with a thick black cell-weighted pooled curve. Version 2 overlays one thin line per sample × celltype, colored by celltype; thick curves are cell-weighted pooled distributions. Pooled curves can broaden due to sample shifts; target sigma refers to within-group circular dispersion. All new outputs are in `run_dir / "qc"`. The added cell_truth column is also saved by the existing final CSV/AnnData export.


In [ ]:
# Ground-truth QC only; no inference, count generation or random-number draws.
phase_delta_dir = run_dir / "qc"
phase_delta_dir.mkdir(parents=True, exist_ok=True)
phase_delta_column = "delta_true_vs_ZT_hour"
phase_required = ["cell_id", "sample_name", "celltype", "true_phase_hour", "ZT", "sigma_bio_hour"]
assert set(phase_required).issubset(cell_truth.columns)
assert cell_truth[phase_required].notna().all().all()
assert np.isfinite(cell_truth[["true_phase_hour", "ZT", "sigma_bio_hour"]].to_numpy(dtype=float)).all()
assert sample_design["sample_name"].is_unique
assert cell_truth["sample_name"].isin(sample_design["sample_name"]).all()
assert cell_truth["celltype"].isin(celltype_names).all()
phase_sample_zt = sample_design.set_index("sample_name")["ZT"]
assert np.allclose(cell_truth["ZT"], cell_truth["sample_name"].map(phase_sample_zt), rtol=0, atol=1e-10)
cell_truth[phase_delta_column] = ((cell_truth["true_phase_hour"] - cell_truth["ZT"] + 12) % 24) - 12
assert cell_truth[phase_delta_column].between(-12, 12, inclusive="left").all()


def summarize_phase_delta(values):
    values = np.asarray(values, dtype=float)
    names = ["circular_mean_delta_hour", "circular_sd_delta_hour", "median_delta_hour",
             "mean_absolute_delta_hour", "q25_delta_hour", "q75_delta_hour"]
    if not len(values):
        return dict.fromkeys(names, np.nan)
    resultant = np.mean(np.exp(1j * values * (2 * np.pi / 24)))
    length = float(np.clip(abs(resultant), 0, 1))
    undefined_direction = length <= 1e-15
    mean = np.nan if undefined_direction else float((np.angle(resultant) * 24 / (2 * np.pi) + 12) % 24 - 12)
    sd = np.inf if undefined_direction else float(np.sqrt(-2 * np.log(length)) * 24 / (2 * np.pi))
    q25, median, q75 = np.quantile(values, [0.25, 0.5, 0.75])
    return dict(zip(names, [mean, sd, median, np.mean(np.abs(values)), q25, q75]))


phase_delta_groups = {(sample, ct): group for (sample, ct), group in
                      cell_truth.groupby(["sample_name", "celltype"], observed=True, sort=False)}
phase_delta_rows = []
for sample in sample_design["sample_name"]:
    for ct in celltype_names:
        group = phase_delta_groups.get((sample, ct))
        values = np.array([]) if group is None else group[phase_delta_column].to_numpy()
        target = float(CONFIG["celltypes"][ct]["sigma_bio_hour"])
        if group is not None:
            assert np.allclose(group["sigma_bio_hour"], target, rtol=0, atol=1e-10)
        phase_delta_rows.append({"sample_name": sample, "ZT": phase_sample_zt.loc[sample],
                                 "celltype": ct, "target_sigma_bio_hour": target,
                                 "n_cells": len(values), **summarize_phase_delta(values)})
sample_celltype_phase_delta_summary = pd.DataFrame(phase_delta_rows)
assert sample_celltype_phase_delta_summary["n_cells"].sum() == len(cell_truth)
sample_celltype_phase_delta_summary.to_csv(
    phase_delta_dir / "sample_celltype_phase_delta_summary.csv", index=False)
phase_export_columns = phase_required + [c for c in
    ["phase_offset_hour", "sample_phase_shift_hour", "phase_center_rad", "phase_center_hour"]
    if c in cell_truth.columns] + [phase_delta_column]
cell_truth[phase_export_columns].to_csv(phase_delta_dir / "cell_true_phase_delta_vs_ZT.csv", index=False)
display(sample_celltype_phase_delta_summary)


In [ ]:
# Common-bin step densities: each sample × celltype curve integrates to one.
from matplotlib.lines import Line2D
phase_delta_edges = np.linspace(-12, 12, int(CONFIG["qc"]["bins"]) + 1)
phase_samples = sample_design["sample_name"].tolist()
phase_sample_colors = plt.get_cmap("turbo")(np.linspace(0.05, 0.95, len(phase_samples)))
phase_ct_colors = plt.get_cmap("tab10")(np.linspace(0, 0.9, len(celltype_names)))


def draw_phase_delta_density(ax, values, **kwargs):
    if len(values):
        density, _ = np.histogram(values, bins=phase_delta_edges, density=True)
        ax.stairs(density, phase_delta_edges, **kwargs)


def save_phase_delta_plot(fig, stem):
    for suffix in ["png", "pdf"]:
        fig.savefig(phase_delta_dir / f"{stem}.{suffix}",
                    dpi=CONFIG["qc"]["dpi"], bbox_inches="tight")
    plt.show()


fig, axes = plt.subplots(1, len(celltype_names), figsize=(4.5 * len(celltype_names), 5),
                         sharex=True, sharey=True, squeeze=False)
for ax, ct in zip(axes.flat, celltype_names):
    for sample, color in zip(phase_samples, phase_sample_colors):
        group = phase_delta_groups.get((sample, ct))
        if group is not None:
            draw_phase_delta_density(ax, group[phase_delta_column], color=color, alpha=0.65, linewidth=1)
    draw_phase_delta_density(ax, cell_truth.loc[cell_truth["celltype"] == ct, phase_delta_column],
                             color="black", linewidth=2.2)
    ax.axvline(0, color="grey", linestyle=":", linewidth=0.8)
    ax.set(title=f"{ct}: target sigma = {CONFIG['celltypes'][ct]['sigma_bio_hour']:g} h",
           xlabel="Single-cell true phase - sample ZT (h)", xlim=(-12, 12), xticks=[-12, -6, 0, 6, 12])
axes[0, 0].set_ylabel("Density (1/h)")
fig.suptitle("True phase vs sample ZT: one sample per thin curve")
handles = [Line2D([0], [0], color=color, label=sample) for sample, color in zip(phase_samples, phase_sample_colors)]
handles.append(Line2D([0], [0], color="black", linewidth=2.2, label="Celltype pooled (cell-weighted)"))
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=min(6, len(handles)), fontsize=8)
save_phase_delta_plot(fig, "08_true_phase_delta_vs_ZT_by_celltype")

fig, ax = plt.subplots(figsize=(10, 6))
for ct, color in zip(celltype_names, phase_ct_colors):
    for sample in phase_samples:
        group = phase_delta_groups.get((sample, ct))
        if group is not None:
            draw_phase_delta_density(ax, group[phase_delta_column], color=color, alpha=0.25, linewidth=0.8)
    draw_phase_delta_density(ax, cell_truth.loc[cell_truth["celltype"] == ct, phase_delta_column],
                             color=color, linewidth=2.4, label=f"{ct} (target sigma {CONFIG['celltypes'][ct]['sigma_bio_hour']:g} h)")
ax.axvline(0, color="grey", linestyle=":", linewidth=0.8)
ax.set(xlabel="Single-cell true phase - sample ZT (h)", ylabel="Density (1/h)",
       xlim=(-12, 12), xticks=[-12, -6, 0, 6, 12],
       title="All sample x celltype distributions together\nThin: individual groups; thick: cell-weighted celltype pooled")
ax.legend()
fig.tight_layout()
save_phase_delta_plot(fig, "09_true_phase_delta_vs_ZT_all_sample_celltype")
print("Ground-truth phase delta QC saved:", phase_delta_dir)


## 5. Genome-wide parameter tables and continuous gene truth

The optional development fallback bootstraps reference mesors/dispersion and rhythmic amplitudes, jitters mesors, assigns uniform phases and sets a configured fraction of genes rhythmic. These are arbitrary development settings, clearly identified in every extra-gene row and in the manifest. They must be replaced before publication. No values from this fallback are labeled empirical.

`true_mesor` is the **log-scale intercept** `a_0`, not the cycle-average count. `true_amplitude` is the log-scale sinusoidal amplitude. `true_phase_rad` is the expression peak relative to true cell phase; it is undefined (NaN) for non-rhythmic genes. The phase of the population-average trajectory versus ZT subtracts the cell-type offset. M0–M7 labels are deliberately absent.

In [ ]:
def validate_params(params_g):
    missing = set(PARAM_COLS) - set(params_g.columns)
    if missing:
        raise ValueError(f"Missing parameter columns: {sorted(missing)}")
    if not params_g.index.is_unique or params_g.index.hasnans:
        raise ValueError("Gene identifiers must be present and unique.")
    if (params_g.index.astype(str).str.strip() == "").any():
        raise ValueError("Empty gene identifier.")
    values = params_g[PARAM_COLS].to_numpy(dtype=float)
    if not np.isfinite(values).all() or (params_g["disp"] <= 0).any():
        raise ValueError("Fourier coefficients must be finite and dispersion positive.")


def get_genome_parameters():
    setting = CONFIG["genome_parameters"]
    assert isinstance(setting["n_genes_total"], int) and setting["n_genes_total"] >= 17
    n_extra = setting["n_genes_total"] - len(params_g_clock)
    if setting["path"]:
        if not setting["provenance"] or not setting["confirmed_full_library_log_offset"]:
            raise ValueError("Provide empirical gene provenance and confirm full-library log offset.")
        path = project_path(setting["path"])
        manifest_path = project_path(setting["manifest_path"])
        empirical_manifest = json.loads(manifest_path.read_text())
        if "full-library offsets" not in empirical_manifest.get("notes", ""):
            raise ValueError("Empirical manifest does not document full-library offsets.")
        if empirical_manifest["fit_settings"]["period"] != CONFIG["period"]:
            raise ValueError("Empirical fit period differs from simulation period.")
        verified_outputs = {}
        for key in ["parameters", "fits", "samples", "pseudobulk"]:
            # Manifest output paths describe the original fitting location.
            # Resolve relocated companion files beside the configured parameter CSV.
            local_path = path if key == "parameters" else path.parent / Path(empirical_manifest["outputs"][key]).name
            digest = file_sha256(local_path)
            if digest != empirical_manifest["outputs"][key + "_sha256"]:
                raise ValueError(f"Empirical {key} hash differs from manifest: {local_path}")
            verified_outputs[key] = {"path": str(local_path), "sha256": digest}
        expected_raw = {Path(r["path"]).name: r["sha256"] for r in empirical_manifest["inputs"]}
        actual_raw = {Path(r["path"]).name: r["sha256"] for r in library_provenance["files"]}
        if expected_raw != actual_raw:
            raise ValueError("Empirical fit inputs differ from the raw UMI library inputs.")
        table = pd.read_csv(path, dtype={"gene_id": str})
        required = {"gene_id", *PARAM_COLS, "parameter_source"}
        if required - set(table.columns):
            raise ValueError(f"Missing empirical CSV columns: {sorted(required - set(table.columns))}")
        for column in PARAM_COLS:
            table[column] = pd.to_numeric(table[column], errors="raise")
        params_g_extra = table.set_index("gene_id")
        validate_params(params_g_extra)
        if "parameter_source" not in params_g_extra or params_g_extra["parameter_source"].isna().any():
            raise ValueError("Every empirical gene needs parameter_source.")
        if params_g_extra["parameter_source"].astype(str).str.strip().eq("").any():
            raise ValueError("Empty parameter_source.")
        if params_g_extra.index.isin(CLOCK_GENES).any():
            raise ValueError("Empirical extra-gene table must exclude the 15 frozen Clock genes.")
        if len(params_g_extra) < n_extra:
            raise ValueError(f"Need {n_extra} extra genes; table has {len(params_g_extra)}.")
        params_g_extra = params_g_extra.iloc[:n_extra].copy()
        params_g_extra["parameter_class"] = "empirical"
        provenance = {"mode": "empirical", "path": str(path), "sha256": file_sha256(path),
                      "notes": setting["provenance"], "selection": "first n_extra rows in file order",
                      "n_available": len(table), "n_selected": n_extra,
                      "confirmed_full_library_log_offset": True,
                      "manifest_path": str(manifest_path), "manifest_sha256": file_sha256(manifest_path),
                      "fit_manifest": empirical_manifest, "verified_outputs": verified_outputs}
    elif setting["allow_development_fallback"]:
        rng = np.random.default_rng(seeds["parameters"])
        fraction = setting["development_rhythmic_fraction"]
        assert 0 < fraction < 1
        assert np.isfinite(setting["development_mesor_sd"]) and setting["development_mesor_sd"] >= 0
        templates = params_g_clock.iloc[rng.integers(0, 15, n_extra)]
        amplitude = np.hypot(templates["a_1"], templates["b_1"]).to_numpy()
        n_rhythmic = min(n_extra - 1, max(1, round(n_extra * fraction)))
        rhythmic = np.zeros(n_extra, dtype=bool)
        rhythmic[rng.choice(n_extra, n_rhythmic, replace=False)] = True
        amplitude[~rhythmic] = 0
        phase = rng.uniform(0, 2 * np.pi, n_extra)
        params_g_extra = pd.DataFrame({
            "a_0": templates["a_0"].to_numpy() + rng.normal(0, setting["development_mesor_sd"], n_extra),
            "a_1": amplitude * np.cos(phase), "b_1": amplitude * np.sin(phase),
            "disp": templates["disp"].to_numpy(),
            "parameter_source": "DEVELOPMENT ONLY: synthetic reference bootstrap",
            "parameter_class": "our extension: development fallback",
        }, index=[f"DEV_gene_{i:05d}" for i in range(n_extra)])
        provenance = {"mode": "DEVELOPMENT_ONLY", "notes": "Not publication parameters"}
        print("DEVELOPMENT ONLY: synthetic extra-gene parameters explicitly enabled.")
    else:
        raise ValueError("Fill empirical genome_parameters.path, or explicitly enable allow_development_fallback for development only.")
    rhythmic = np.hypot(params_g_extra["a_1"], params_g_extra["b_1"]) > 0
    if rhythmic.all() or not rhythmic.any():
        raise ValueError("Selected extra genes must include rhythmic and non-rhythmic genes.")
    params_g_seed = params_g_clock.copy()
    params_g_seed["parameter_source"] = "scRITMO-original: ccg_zhang_context.csv"
    params_g_seed["parameter_class"] = "scRITMO-original"
    params_g = pd.concat([params_g_seed, params_g_extra])
    params_g.index.name = "gene_id"
    validate_params(params_g)
    return params_g, provenance


def get_params_by_celltype(params_g):
    perturb_cols = ["delta_mesor", "amplitude_scale", "amplitude_add", "phase_shift_hour"]
    perturbations = pd.DataFrame(columns=["gene_id", "celltype", *perturb_cols])
    provenance = {"mode": "none; no gene-level cell-type perturbations"}
    if CONFIG["perturbations_path"]:
        if not CONFIG["perturbations_provenance"]:
            raise ValueError("Provide perturbations_provenance in CONFIG.")
        path = project_path(CONFIG["perturbations_path"])
        perturbations = pd.read_csv(path, dtype={"gene_id": str, "celltype": str})
        assert not perturbations.duplicated(["gene_id", "celltype"]).any()
        assert perturbations["gene_id"].isin(params_g.index).all()
        assert perturbations["celltype"].isin(celltype_names).all()
        assert not perturbations["gene_id"].isin(CLOCK_GENES).any(), "Clock parameters are frozen."
        assert np.isfinite(perturbations[perturb_cols].to_numpy(dtype=float)).all()
        assert (perturbations[["amplitude_scale", "amplitude_add"]] >= 0).all().all()
        provenance = {"path": str(path), "sha256": file_sha256(path),
                      "notes": CONFIG["perturbations_provenance"], "mode": "our extension"}
    params_by_celltype, truth = {}, []
    for ct in celltype_names:
        params = params_g.copy(deep=True)
        rows = perturbations.loc[perturbations["celltype"] == ct]
        for row in rows.itertuples(index=False):
            base = params_g.loc[row.gene_id]
            amp = np.hypot(base.a_1, base.b_1) * row.amplitude_scale + row.amplitude_add
            phase = np.arctan2(base.b_1, base.a_1) + row.phase_shift_hour * w
            params.loc[row.gene_id, ["a_0", "a_1", "b_1"]] = [
                base.a_0 + row.delta_mesor, amp * np.cos(phase), amp * np.sin(phase)]
        # Exact equality: never reconstruct Fourier coefficients for frozen clock rows.
        pd.testing.assert_frame_equal(params.loc[CLOCK_GENES, params_g_clock.columns],
                                      params_g_clock.rename_axis("gene_id"), check_exact=True)
        validate_params(params)
        params_by_celltype[ct] = params
        df = params[[*PARAM_COLS, "parameter_source", "parameter_class"]].copy()
        df["celltype"] = ct
        df["is_seed"] = df.index.isin(CLOCK_GENES)
        df["true_mesor"] = df["a_0"]
        df["true_amplitude"] = np.hypot(df["a_1"], df["b_1"])
        df["true_rhythmic"] = df["true_amplitude"] > 0
        df["true_phase_rad"] = np.where(df["true_rhythmic"],
            np.arctan2(df["b_1"], df["a_1"]) % (2 * np.pi), np.nan)
        df["true_phase_hour"] = df["true_phase_rad"] * rh
        df["phase_vs_ZT_hour"] = (df["true_phase_hour"] - CONFIG["celltypes"][ct]["phase_offset_hour"]) % 24
        df["perturbation_applied"] = df.index.isin(rows["gene_id"])
        df["perturbation_source"] = np.where(df["perturbation_applied"], "our extension: parameter table", "none")
        truth.append(df.reset_index())
    return params_by_celltype, pd.concat(truth, ignore_index=True), perturbations, provenance


params_g, gene_provenance = get_genome_parameters()
params_by_celltype, gene_truth, perturbations, perturbation_provenance = get_params_by_celltype(params_g)
for ct in celltype_names:
    pd.testing.assert_frame_equal(params_by_celltype[ct], params_g, check_exact=True)
assert len(gene_truth) == len(params_g) * len(celltype_names)
display(gene_truth.groupby(["celltype", "parameter_class", "true_rhythmic"]).size().rename("n_genes"))

## 6. Counts through the official generator

The wrapper iterates cell types and bounded cell chunks, passing complete gene tables to the official generator. No NB sampling kernel is reimplemented. Dispersion is passed in gene order; row alignment is restored to `cell_truth`. No normalization or fixed-row-sum correction is applied.

The model mean is `s_i exp(a_0 + a_1 cos θ_i + b_1 sin θ_i)`. The QC expectation uses this equation only to draw trajectories and check numerical range. **Simulated row sums need not match input library sizes**, especially when the simulated panel is a subset of the genome. Downstream methods should use `obs['library_size']` for the generative exposure; re-estimating it from panel sums changes the model.

In [ ]:
def generate_counts(cell_truth, params_by_celltype):
    torch.manual_seed(seeds["counts"])
    blocks, row_order = [], []
    for ct in celltype_names:
        params = params_by_celltype[ct]
        indices = np.flatnonzero(cell_truth["celltype"].to_numpy() == ct)
        for start in range(0, len(indices), CONFIG["cell_chunk"]):
            idx = indices[start:start + CONFIG["cell_chunk"]]
            df_obs_group = cell_truth.iloc[idx]
            # Guard float32 NB probability saturation before official generation.
            max_log_mu = np.log(df_obs_group["library_size"].max()) + params["a_0"] + np.hypot(params["a_1"], params["b_1"])
            if (max_log_mu + np.log(params["disp"]) > np.log(1e6)).any():
                raise ValueError("Parameters risk float32 NB saturation; check units and empirical table.")
            generated_data = simulate_data_no_context(
                phases=df_obs_group["true_phase_rad"].to_numpy(),
                seq_depths=df_obs_group["library_size"].to_numpy(),
                fourier_coefficients=params,
                context_label=ct,
                noise_model="nb",
                dispersion=params["disp"].to_numpy(),
            )
            assert generated_data.shape == (len(idx), len(params_g))
            assert np.isfinite(generated_data).all() and (generated_data >= 0).all()
            assert np.equal(generated_data, np.floor(generated_data)).all()
            blocks.append(sparse.csr_matrix(generated_data.astype(np.int64)))
            row_order.extend(idx)
    assert sorted(row_order) == list(range(len(cell_truth)))
    return sparse.vstack(blocks, format="csr")[np.argsort(row_order)]


data = generate_counts(cell_truth, params_by_celltype)
cell_truth["simulated_gene_count_sum"] = np.asarray(data.sum(axis=1)).ravel()
cell_truth["zero_fraction"] = 1 - np.asarray(data.getnnz(axis=1)).ravel() / data.shape[1]
print("Generated cells x genes:", data.shape)

## 7. Pairwise continuous truth

One row per gene and unordered cell-type pair, oriented `a → b`. Differences mean **b − a**. Phase differences wrap to [−π, π), or [−12, 12) hours. Positive intrinsic phase difference means a later peak relative to cell phase. Undefined phase differences (either gene non-rhythmic) and amplitude ratios with a zero denominator are NaN, not zero. Both intrinsic and external-ZT phase differences are retained. No M0–M7 labels or arbitrary classification thresholds are introduced.

In [ ]:
def circular_phase_diff_rad(phi2, phi1):
    return (phi2 - phi1 + np.pi) % (2 * np.pi) - np.pi


def make_pairwise_truth(gene_truth):
    pairs = []
    columns = ["true_mesor", "true_amplitude", "true_phase_rad", "true_rhythmic", "phase_vs_ZT_hour"]
    for ct_a, ct_b in itertools.combinations(celltype_names, 2):
        a = gene_truth.loc[gene_truth["celltype"] == ct_a].set_index("gene_id")
        b = gene_truth.loc[gene_truth["celltype"] == ct_b].set_index("gene_id").loc[a.index]
        df = a[columns].join(b[columns], lsuffix="_a", rsuffix="_b")
        df["celltype_a"], df["celltype_b"] = ct_a, ct_b
        df["is_seed"] = a["is_seed"]
        df["delta_mesor"] = df["true_mesor_b"] - df["true_mesor_a"]
        df["delta_amplitude"] = df["true_amplitude_b"] - df["true_amplitude_a"]
        df["amplitude_ratio"] = df["true_amplitude_b"] / df["true_amplitude_a"].replace(0, np.nan)
        df["delta_phase_rad"] = circular_phase_diff_rad(df["true_phase_rad_b"], df["true_phase_rad_a"])
        df["delta_phase_hour"] = df["delta_phase_rad"] * rh
        df["delta_phase_vs_ZT_hour"] = circular_phase_diff_rad(
            df["phase_vs_ZT_hour_b"] * w, df["phase_vs_ZT_hour_a"] * w) * rh
        pairs.append(df.reset_index())
    return pd.concat(pairs, ignore_index=True)


pairwise_truth = make_pairwise_truth(gene_truth)
assert len(pairwise_truth) == len(params_g) * len(list(itertools.combinations(celltype_names, 2)))
assert not gene_truth.duplicated(["gene_id", "celltype"]).any()
assert pairwise_truth.loc[pairwise_truth["is_seed"], "delta_mesor"].eq(0).all()
assert pairwise_truth.loc[pairwise_truth["is_seed"], "delta_amplitude"].eq(0).all()
display(pairwise_truth.head())

## 8. Simulation QC

Plots and numeric summaries are saved with the output. These diagnose the simulation, not acceptance criteria selected after seeing method performance. Composition is expected to fluctuate; library distributions match by bootstrap rather than exactly; the mean–variance plot pools heterogeneous cell means, so a single homogeneous NB curve is not an appropriate reference.

In [ ]:
qc_dir = run_dir / "qc"
qc_dir.mkdir(exist_ok=True)


def save_qc(fig, name):
    fig.tight_layout()
    fig.savefig(qc_dir / f"{name}.png", dpi=CONFIG["qc"]["dpi"], bbox_inches="tight")
    plt.show()


# 1. Empirical exposures versus simulated exposures; panel totals are distinct.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for values, label in [(library_pool["library_size"], "GSE145197 pooled"),
                       (cell_truth["library_size"], "Simulated exposure")]:
    sorted_values = np.sort(values)
    axes[0].plot(sorted_values, np.arange(1, len(values) + 1) / len(values), label=label)
axes[0].set(xscale="log", xlabel="Full-library size", ylabel="ECDF")
axes[0].legend()
axes[1].scatter(cell_truth["library_size"], cell_truth["simulated_gene_count_sum"], s=5, alpha=0.3)
axes[1].set(xlabel="Input full-library exposure", ylabel="Simulated gene-panel count sum")
save_qc(fig, "01_library_sizes")
library_summary = pd.DataFrame({
    "empirical": library_pool["library_size"].describe(),
    "simulated_exposure": cell_truth["library_size"].describe(),
    "simulated_panel_sum": cell_truth["simulated_gene_count_sum"].describe(),
})
library_summary.to_csv(qc_dir / "library_summary.csv")
ks_statistic = float(ks_2samp(library_pool["library_size"], cell_truth["library_size"]).statistic)
print("Descriptive ECDF KS distance (bootstrap samples are not independent):", ks_statistic)

# 2. Random cell-type composition, including zero-size groups.
composition = pd.crosstab(cell_truth["sample_name"], cell_truth["celltype"]).reindex(
    index=sample_design["sample_name"], columns=celltype_names, fill_value=0)
fractions = composition.div(composition.sum(axis=1), axis=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fractions.plot.bar(stacked=True, ax=axes[0], legend=False)
axes[0].set(ylabel="Cell-type fraction", xlabel="Biological sample")
for ct, p in zip(celltype_names, probabilities):
    axes[1].scatter(np.repeat(ct, len(fractions)), fractions[ct], alpha=0.5)
    axes[1].scatter([ct], [p], marker="_", s=350, color="black")
axes[1].set(ylabel="Observed fraction (black = target)")
save_qc(fig, "02_celltype_composition")
composition.to_csv(qc_dir / "celltype_counts.csv")
print("Empty sample x cell-type groups:", int((composition == 0).sum().sum()))

# 3. Phase distribution by ZT, cell type and centered residual desynchrony.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(cell_truth["ZT"], cell_truth["true_phase_hour"], s=4, alpha=0.15)
axes[0].set(xlabel="ZT (hour)", ylabel="True phase (hour)", ylim=(0, 24))
phase_summary = []
for ct in celltype_names:
    df = cell_truth.loc[cell_truth["celltype"] == ct]
    residual = circular_phase_diff_rad(df["true_phase_rad"], df["phase_center_rad"])
    axes[1].hist(df["true_phase_hour"], bins=np.linspace(0, 24, CONFIG["qc"]["bins"] + 1),
                 histtype="step", density=True, label=ct)
    axes[2].hist(residual * rh, bins=np.linspace(-12, 12, CONFIG["qc"]["bins"] + 1),
                 histtype="step", density=True, label=ct)
    phase_summary.append({"celltype": ct, "n_cells": len(df),
                          "target_sigma_bio_hour": CONFIG["celltypes"][ct]["sigma_bio_hour"],
                          "observed_residual_cstd_hour": float(circstd(residual) * rh) if len(df) else np.nan})
axes[1].set(xlabel="True phase (hour)", ylabel="Density")
axes[2].set(xlabel="Phase minus sample/type center (hour)", ylabel="Density")
axes[1].legend()
save_qc(fig, "03_phases")
pd.DataFrame(phase_summary).to_csv(qc_dir / "phase_summary.csv", index=False)

In [ ]:
# 4. Parameter distributions, with flat-gene phase excluded.
# These four panels are gene metrics, not cell types; each overlays all configured CTs.
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ct in celltype_names:
    df = gene_truth.loc[gene_truth["celltype"] == ct]
    for ax, column in zip(axes, ["true_mesor", "true_amplitude", "true_phase_hour", "disp"]):
        ax.hist(df[column].dropna(), bins=CONFIG["qc"]["bins"], histtype="step", label=ct)
        ax.set(xlabel=column, ylabel="Genes")
axes[0].legend()
save_qc(fig, "04_gene_parameters")

# 5. Zero fractions by cell and gene, plus cell-type mean/variance.
gene_qc_rows = []
for ct in celltype_names:
    matrix = data[cell_truth["celltype"].to_numpy() == ct].astype(np.float64)
    if matrix.shape[0] == 0:
        continue
    mean_g = np.asarray(matrix.mean(axis=0)).ravel()
    var_g = np.maximum(0, np.asarray(matrix.power(2).mean(axis=0)).ravel() - mean_g**2)
    gene_qc_rows.append(pd.DataFrame({"gene_id": params_g.index, "celltype": ct,
        "mean_count": mean_g, "variance_count_ddof0": var_g,
        "zero_fraction": 1 - matrix.getnnz(axis=0) / matrix.shape[0]}))
gene_qc = pd.concat(gene_qc_rows, ignore_index=True)
gene_qc.to_csv(qc_dir / "gene_count_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(cell_truth["zero_fraction"], bins=CONFIG["qc"]["bins"])
axes[0].set(xlabel="Cell zero fraction", ylabel="Cells")
for ct in celltype_names:
    axes[1].hist(gene_qc.loc[gene_qc["celltype"] == ct, "zero_fraction"],
                 bins=CONFIG["qc"]["bins"], histtype="step", label=ct)
axes[1].set(xlabel="Gene zero fraction", ylabel="Genes")
axes[1].legend()
save_qc(fig, "05_zero_fraction")

# 6. Mean-variance relationship (pooled heterogeneity inflates variance).
fig, ax = plt.subplots(figsize=(6, 4))
for ct in celltype_names:
    df = gene_qc.loc[(gene_qc["celltype"] == ct) & (gene_qc["mean_count"] > 0)
                    & (gene_qc["variance_count_ddof0"] > 0)]
    ax.scatter(df["mean_count"], df["variance_count_ddof0"], s=8, alpha=0.5, label=ct)
ax.set(xscale="log", yscale="log", xlabel="Mean count", ylabel="Variance (ddof=0)")
ax.legend()
save_qc(fig, "06_mean_variance")

# 7. Representative trajectories: normalize by known exposure, never by panel sum.
trajectory_genes = CONFIG["qc"]["trajectory_genes"]
assert trajectory_genes and set(trajectory_genes).issubset(params_g.index)
fig, axes = plt.subplots(len(trajectory_genes), len(celltype_names),
                         figsize=(4 * len(celltype_names), 3 * len(trajectory_genes)), squeeze=False)
phi_grid = np.linspace(0, 2 * np.pi, 241)
for j, ct in enumerate(celltype_names):
    idx = np.flatnonzero(cell_truth["celltype"].to_numpy() == ct)
    if len(idx) > CONFIG["qc"]["trajectory_cells"]:
        idx = idx[np.linspace(0, len(idx) - 1, CONFIG["qc"]["trajectory_cells"]).astype(int)]
    for i, gene in enumerate(trajectory_genes):
        p = params_by_celltype[ct].loc[gene]
        y = data[idx, params_g.index.get_loc(gene)].toarray().ravel()
        exposure = cell_truth.iloc[idx]["library_size"].to_numpy()
        axes[i, j].scatter(cell_truth.iloc[idx]["true_phase_hour"], y / exposure * 1e4,
                           s=5, alpha=0.2)
        expected = np.exp(p.a_0 + p.a_1 * np.cos(phi_grid) + p.b_1 * np.sin(phi_grid)) * 1e4
        axes[i, j].plot(phi_grid * rh, expected, color="black")
        axes[i, j].set(title=f"{gene} / {ct}", xlabel="True cell phase (hour)", ylabel="Count / exposure x 10,000")
save_qc(fig, "07_trajectories")

In [ ]:
output_mode = gene_provenance["mode"]
adata_sim = anndata.AnnData(
    X=data,
    obs=cell_truth.set_index("cell_id").copy(),
    var=params_g[["parameter_source", "parameter_class"]].copy(),
)
adata_sim.var["is_seed"] = adata_sim.var_names.isin(CLOCK_GENES)
adata_sim.uns["simulation_mode"] = output_mode
adata_sim.uns["library_size_semantics"] = "Pooled GSE145197 full-library exposure; not X.sum(axis=1)"
adata_sim.uns["gene_truth_file"] = "gene_truth.csv"
adata_sim.uns["pairwise_truth_file"] = "pairwise_truth.csv"
adata_sim.uns["scRITMO_commit"] = scritmo_commit
adata_sim.uns["config_json"] = json.dumps(CONFIG, sort_keys=True)

for name, table in [("cell_truth", cell_truth), ("gene_truth", gene_truth),
                     ("sample_design", sample_design), ("pairwise_truth", pairwise_truth),
                     ("perturbations_applied", perturbations)]:
    table.to_csv(run_dir / f"{name}.csv", index=False)
params_g.to_csv(run_dir / "baseline_gene_parameters.csv")
adata_sim.write_h5ad(run_dir / "scRITMO_extended_simulation.h5ad")

# Round-trip alignment and count integrity.
check = anndata.read_h5ad(run_dir / "scRITMO_extended_simulation.h5ad")
assert check.obs_names.tolist() == cell_truth["cell_id"].tolist()
assert check.var_names.tolist() == params_g.index.tolist()
assert (check.X != data).nnz == 0
assert np.array_equal(check.obs["library_size"], cell_truth["library_size"])
del check

manifest = {
    "schema_version": "1.0", "created_utc": datetime.now(timezone.utc).isoformat(),
    "mode": output_mode, "publication_parameters_available": output_mode == "empirical",
    "config": CONFIG, "seed_streams": seeds,
    "scRITMO": {"repo": str(repo), "commit": scritmo_commit, "working_tree_status": scritmo_status,
                 "source_files": source_info, "uploaded_source_hashes": SOURCE_HASHES,
                 "clock_parameters": {"path": str(clock_path), "sha256": CLOCK_HASH}},
    "inputs": {"library": library_provenance, "extra_genes": gene_provenance,
               "perturbations": perturbation_provenance},
    "environment": {"python": sys.version, "platform": platform.platform(),
                    "packages": {name: version(name) for name in
                                 ["torch", "numpy", "pandas", "scipy", "anndata", "scanpy", "scikit-learn", "matplotlib"]}},
    "shape": {"n_cells": data.shape[0], "n_genes": data.shape[1], "n_samples": len(sample_design)},
    "semantics": {"generator": "official simulate_data_no_context -> generate_nb_data; CPU; NB2",
                  "dispersion": "gene-wise disp array explicitly passed, Var=mu+disp*mu^2",
                  "mesor": "natural-log intercept with full-library exposure",
                  "phase": "intrinsic peak radians, modulo 2*pi; undefined for non-rhythmic genes",
                  "sigma_bio": "circular standard deviation in hours; exact numerical VM inversion",
                  "pairwise": "b-a; wrapped phase differences; no M0-M7 labels",
                  "replicates": "independent animals at each ZT; optional shared sample shift",
                  "library_size": "bootstrap full-library exposure, not simulated gene-panel count sum"},
    "qc": {"library_ecdf_ks_distance": ks_statistic,
           "empty_sample_celltype_groups": int((composition == 0).sum().sum()),
           "overall_zero_fraction": float(1 - data.nnz / (data.shape[0] * data.shape[1]))},
    "outputs": {str(p.relative_to(run_dir)): file_sha256(p) for p in sorted(run_dir.rglob("*"))
                if p.is_file() and p.name != "simulation_manifest.json"},
}
(run_dir / "simulation_manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False))
print("Saved:", run_dir)
print("Mode:", output_mode)
print("Clock parameters: unchanged; h5ad round-trip: passed")

In [ ]:
from scipy.io import mmwrite
r_input_dir = run_dir / "R_input"
r_input_dir.mkdir(exist_ok=True)
mmwrite(r_input_dir / "count_gene_by_cell.mtx", data.T)
pd.DataFrame({"gene": params_g.index.astype(str)}).to_csv(r_input_dir / "genes.csv", index=False)
metadata_R = cell_truth[["cell_id", "sample_name", "ZT", "celltype", "sigma_bio_hour", "true_phase_hour", "true_phase_rad"]].copy()
metadata_R = metadata_R.rename(columns={"cell_id": "cell", "true_phase_hour": "T_true", "true_phase_rad": "Theta_true"})
assert set(metadata_R["celltype"]) == set(celltype_names)
expected_metadata_sigma = metadata_R["celltype"].map(
    {ct: CONFIG["celltypes"][ct]["sigma_bio_hour"] for ct in celltype_names})
assert np.array_equal(metadata_R["sigma_bio_hour"], expected_metadata_sigma)
assert metadata_R["cell"].tolist() == cell_truth["cell_id"].tolist()
metadata_R.to_csv(r_input_dir / "metadata.csv", index=False)
assert data.T.shape == (1000, 9000)
assert len(metadata_R) == 9000
assert metadata_R["cell"].is_unique
print("R input saved:", r_input_dir)
print("count:", data.T.shape, "genes x cells")
display(metadata_R.head())